# Intent classification (LLM-based)

Classify intent (category + subcategory) using the same LLM parallel/cache approach as commercial vertical extraction. Uses **query only** (first user message). Taxonomy matches [intent_classification.ipynb](intent_classification.ipynb): informational, navigational, commercial_investigation, transactional and their sub-categories.

- **Data:** English-only from `data/english_chunks/` (no sampling). Optionally process an index range.
- **Output:** Parquets preserve all original columns; new columns `intent_major`, `intent_sub` added.

In [1]:
# Control variables (edit and run first)
ENGLISH_CHUNKS_DIR = "data/english_chunks"
INTENT_OUTPUT_DIR = "intent_output"
PROCESS_ALL = True   # if False, use index range below
START_INDEX = 20000
END_INDEX = 50000   # used only when PROCESS_ALL is False
SAVE_BY_CATEGORY = True
SAVE_CHUNK_SIZE = 50000   # chunk full-table parquet when rows > this (0 = single file)
INTENT_LLM_CACHE_PATH = "intent_output/intent_llm.parquet"
USE_CACHE = True       # set False to re-run LLM for all rows (ignore existing cache)
LLM_BATCH_SIZE = 100
LLM_MAX_WORKERS = 100
SHOW_PROGRESS = True   # progress bar during LLM run

# Manual spot-check (full conversation): set category, optional sub, and how many to show
SPOTCHECK_MAJOR = "informational"   # intent_major: informational, navigational, commercial_investigation, transactional
SPOTCHECK_SUB = None                # intent_sub, or None for any
SPOTCHECK_N = 3                     # number of conversations to print
SPOTCHECK_WIDTH = 80                # textwrap width for conversation text

## Load data

Load from english_chunks (no sampling), add first user message as `text`, drop empty, then optionally slice by index range.

In [2]:
from pathlib import Path

from eda_utils import ensure_conversation_parsed, load_english_chunked_parquet
from insights_utils import extract_text_column

chunks_dir = Path(ENGLISH_CHUNKS_DIR)
if not chunks_dir.exists() or not list(chunks_dir.glob("english_*.parquet")):
    raise FileNotFoundError(f"No english_*.parquet found in {chunks_dir}. Run Export English-only in explore.ipynb first.")

df_full = load_english_chunked_parquet(chunks_dir)
df_full = ensure_conversation_parsed(df_full)
df_full["text"] = extract_text_column(df_full, mode="first_user", conversation_col="conversation")
df = df_full[df_full["text"].fillna("").astype(str).str.strip() != ""].copy().reset_index(drop=True)

if PROCESS_ALL:
    df_to_process = df
else:
    df_to_process = df.iloc[START_INDEX:END_INDEX].copy()

print(f"Loaded {len(df)} rows with non-empty query; processing {len(df_to_process)} rows.")
print(f"Columns: {list(df_to_process.columns)}")
df_to_process.head(2)

Loaded 283291 rows with non-empty query; processing 283291 rows.
Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'text']


,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,text
0,26c5dc109920789f9199ff9b37acb8c1,gpt-4,2023-04-10 00:01:08+00:00,"[{'content': 'Write a very long, elaborate, de...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0001022904907586053, 'i...",False,False,"Write a very long, elaborate, descriptive and ..."
1,e87a1aeb9aafa35c00da39ddeb1139a0,gpt-4,2023-04-10 00:01:10+00:00,"[{'content': 'what are you?', 'language': 'Eng...",1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 9.160337503999472e-05, 'i...",False,False,what are you?


## Run LLM intent classification

Call OpenAI in parallel (batched, cached). Adds `intent_major` and `intent_sub` to the dataframe.

In [3]:
from dotenv import load_dotenv
load_dotenv()

from commercial_vertical_utils import label_intent_llm_parallel

df_result = label_intent_llm_parallel(
    df_to_process,
    text_col="text",
    cache_path=INTENT_LLM_CACHE_PATH,
    use_cache=USE_CACHE,
    batch_size=LLM_BATCH_SIZE,
    max_workers=LLM_MAX_WORKERS,
    show_progress=SHOW_PROGRESS,
)

print("Intent major distribution:")
display(df_result["intent_major"].value_counts().to_frame("count"))
print("Intent sub distribution:")
display(df_result["intent_sub"].value_counts().to_frame("count"))

/Users/Larry.Jin/miniconda3/envs/wildchat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Intent LLM: 100%|██████████| 2333/2333 [3:59:09<00:00,  6.15s/batch]  


Note: OpenAI rate limit (429) was hit for 311 request(s); retried with backoff. Consider lowering max_workers if this is frequent.
Intent major distribution:


,count
intent_major,
informational,274480
commercial_investigation,4112
navigational,3693
transactional,1006


Intent sub distribution:


,count
intent_sub,
creative_writing,130455
casual_other,54774
education,50217
coding,23774
support,15260
commercial_product,4112
navigational,3693
transactional,1006


## Save parquets

Save the processed table (all original columns + `intent_major`, `intent_sub`). Optionally write per-category subsets under `by_major/` and `by_sub/`.

In [4]:
from pathlib import Path

from insights_utils import ensure_output_dir
from intent_taxonomy import save_classified_by_category

# Put output in a subfolder whose name reflects the index range (or "all")
if PROCESS_ALL:
    range_label = "all"
    full_name = "intent_classified_llm_all.parquet"
else:
    range_label = f"{START_INDEX}_{END_INDEX}"
    full_name = f"intent_classified_llm_{START_INDEX}_{END_INDEX}.parquet"

out_dir = Path(INTENT_OUTPUT_DIR) / range_label
ensure_output_dir(out_dir)

if SAVE_BY_CATEGORY and "intent_major" in df_result.columns:
    save_classified_by_category(df_result, out_dir, full_name=full_name, chunk_size=SAVE_CHUNK_SIZE if SAVE_CHUNK_SIZE else None)
    chunked = SAVE_CHUNK_SIZE and len(df_result) > SAVE_CHUNK_SIZE
    msg = f"Saved full table (chunks of {SAVE_CHUNK_SIZE})" if chunked else "Saved full table"
    print(f"{msg} and per-category parquet files under {out_dir}/.")
else:
    out_path = out_dir / full_name
    df_result.to_parquet(out_path, index=False)
    print(f"Saved {len(df_result)} rows to {out_path}.")

Saved full table (chunks of 50000) and per-category parquet files under intent_output/all/.


## Spot-check intent category

Sample a few rows per `intent_major` to visually verify labels (query text + intent_major + intent_sub).

In [5]:
import pandas as pd

def spotcheck_intent(df: pd.DataFrame, n_per_category: int = 5, text_col: str = "text", seed: int = 42) -> pd.DataFrame:
    """Sample n_per_category rows per intent_major for spot-checking."""
    if "intent_major" not in df.columns:
        return pd.DataFrame()
    samples = []
    for major in df["intent_major"].dropna().unique():
        subset = df[df["intent_major"] == major]
        n = min(n_per_category, len(subset))
        if n > 0:
            sampled = subset.sample(n=n, random_state=seed)
            samples.append(sampled[[text_col, "intent_major", "intent_sub"]].copy())
    if not samples:
        return pd.DataFrame()
    return pd.concat(samples, ignore_index=True)

spot = spotcheck_intent(df_result, n_per_category=5)
print("Spot-check sample (query text truncated to 200 chars):")
spot["text_preview"] = spot["text"].fillna("").astype(str).str[:200]
display(spot[["text_preview", "intent_major", "intent_sub"]])

Spot-check sample (query text truncated to 200 chars):


,text_preview,intent_major,intent_sub
0,Kindly write a letter regarding of allowing me...,informational,creative_writing
1,Create new Excel Table from data: Location\tID...,informational,coding
2,Can you rewrite a text for me but using simple...,informational,creative_writing
3,provide me accurate and comprehensive answers ...,informational,education
4,(In the clubroom…)\n\nSayori: (talking about Y...,informational,casual_other
5,Franklin bought the whistle\n\nfrom his cousin...,transactional,transactional
6,Multi brand store promotion with flat 40% on e...,transactional,transactional
7,formal email to bank for bank account transfer...,transactional,transactional
8,"Could you give me only the ip address, usernam...",transactional,transactional
9,"revised\n\nDear Ms. Ehsan, Hope you're doing w...",transactional,transactional


## Manual spot-check: full conversation

Filter by **category** (and optionally **sub-category**), then show **N** full conversations with text-wrapped content. Set `SPOTCHECK_MAJOR`, `SPOTCHECK_SUB` (or `None` for any), and `SPOTCHECK_N` in the control cell at the top.

In [6]:
from intent_analysis_utils import ensure_conversation_normalized, format_conversation

df_spot = ensure_conversation_normalized(df_result, conversation_col="conversation")
mask = df_spot["intent_major"] == SPOTCHECK_MAJOR
if SPOTCHECK_SUB is not None:
    mask = mask & (df_spot["intent_sub"] == SPOTCHECK_SUB)
subset = df_spot.loc[mask]
n_show = min(SPOTCHECK_N, len(subset))
if n_show == 0:
    print(f"No rows for intent_major={SPOTCHECK_MAJOR!r}, intent_sub={SPOTCHECK_SUB!r}. Check SPOTCHECK_MAJOR / SPOTCHECK_SUB.")
else:
    sample = subset.sample(n=n_show, random_state=42)
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print("=" * 60)
        print(f"Conversation {i}/{n_show} | {row['conversation_id']} | {row['intent_major']} | {row['intent_sub']}")
        print("=" * 60)
        print(format_conversation(row["conversation"], width=SPOTCHECK_WIDTH))
        print()

Conversation 1/3 | f0beead404dd10ad8c78135fad47240b | informational | creative_writing
[user]
Kindly write a letter regarding of allowing me to create a new account on
HelloTalk because of the issue I've made on my previous account "id_kin26" make
it emphatic

[assistant]
[Your Name] [Your Address] [City, State, ZIP] [Email Address] [Date]  HelloTalk
Support Team [HelloTalk Address] City, State, ZIP  Subject: Request for
Permission to Create a New HelloTalk Account  Dear HelloTalk Support Team,  I
hope this letter finds you in good health and high spirits. I am writing to
request your kind assistance in allowing me to create a new account on
HelloTalk. I deeply regret the unfortunate issue I caused during my previous
account usage under the username "id_kin26."  Firstly, I would like to express
my sincere apologies for any inconvenience or disruption my actions may have
caused. It was never my intention to create any problems or violate any rules
and regulations set forth by HelloTalk.